# 204. Contrastive Decoding：expert/amateur 与 plausibility constraint 怎样实现？

> **面试问题：怎样对齐两个模型的 token 概率，过滤 expert 不认可的候选，计算对比分数，并验证 CD 的质量、成本和失败边界？**

## 先给结论

这类题要把论文概念拆为可判定的数学/状态合同：方向从什么对比样本估计、解码如何保留合法候选、熵统计的概率空间是什么、权重编辑如何验证 rewrite/generalization/locality。教学实现用受控小向量，不代表真实模型安全、语言质量或跨领域泛化。

## 一手资料

- [Contrastive Decoding](https://arxiv.org/abs/2210.15097)
- [Contrastive Search vs. Decoding](https://arxiv.org/abs/2211.10797)
- [Holtzman et al. Degeneration](https://arxiv.org/abs/1904.09751)

In [ ]:
notebook_contract = {"mode": "small-controlled-arrays", "oracle": "assertions", "production": "needs-evaluation-and-versioning"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "small-controlled-arrays"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：Contrastive Decoding 比较 expert 与 amateur

CD 不等于拿两个模型投票。它以 expert 的条件概率保证可读性，再用 expert 与较弱 amateur 的 log 概率差来惩罚后者特别偏好的退化 token。两模型必须共享词表和 chat template 才能逐 token 对齐。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
expert = {"the": 0.50, "river": 0.30, "the the": 0.20}  # 执行本行的状态、计算或校验逻辑。
amateur = {"the": 0.70, "river": 0.05, "the the": 0.25}  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(sum(expert.values()), 1.0)  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(sum(amateur.values()), 1.0)  # 执行本行的状态、计算或校验逻辑。
assert set(expert) == set(amateur)  # 执行本行的状态、计算或校验逻辑。


## 2. plausibility constraint：先过滤 expert 不认可的候选

若只取概率差，低概率垃圾 token 可能因 amateur 更不喜欢而被选中。最小约束是仅保留 expert 概率超过其最大概率某个比例的 token；阈值应版本化并通过生成质量/多样性评测选择。


In [ ]:
def plausible_tokens(expert, alpha):  # 执行本行的状态、计算或校验逻辑。
    maximum = max(expert.values())  # 执行本行的状态、计算或校验逻辑。
    return {token for token, probability in expert.items() if probability >= alpha * maximum}  # 执行本行的状态、计算或校验逻辑。
plausible = plausible_tokens(expert, 0.4)  # 执行本行的状态、计算或校验逻辑。
assert plausible == {"the", "river", "the the"}  # 执行本行的状态、计算或校验逻辑。
assert plausible_tokens(expert, 0.7) == {"the"}  # 执行本行的状态、计算或校验逻辑。
assert "river" in plausible_tokens(expert, 0.5)  # 执行本行的状态、计算或校验逻辑。


## 3. 对比分数：计算 log p_expert - beta log p_amateur

对数空间避免连乘下溢，也让 beta 可调节 amateur 惩罚。所有概率必须为正；真实模型通常在 logits 上以稳定 log-softmax 实现，并要考虑 tokenization 和 batch 对齐。


In [ ]:
def contrastive_score(token, expert, amateur, beta):  # 执行本行的状态、计算或校验逻辑。
    return math.log(expert[token]) - beta * math.log(amateur[token])  # 执行本行的状态、计算或校验逻辑。
scores = {token: contrastive_score(token, expert, amateur, 1.0) for token in expert}  # 执行本行的状态、计算或校验逻辑。
assert scores["river"] > scores["the"]  # 执行本行的状态、计算或校验逻辑。
assert scores["river"] > scores["the the"]  # 执行本行的状态、计算或校验逻辑。
assert all(math.isfinite(value) for value in scores.values())  # 执行本行的状态、计算或校验逻辑。


## 4. 选择：只在 plausible set 内取最大分数

选择逻辑要把候选过滤和排序分开，便于诊断候选被删掉还是排序输掉。若过滤集为空，不能静默返回任意 token；可按策略回退到 expert greedy 或报错并结束。


In [ ]:
def choose_token(expert, amateur, beta, alpha):  # 执行本行的状态、计算或校验逻辑。
    candidates = plausible_tokens(expert, alpha)  # 执行本行的状态、计算或校验逻辑。
    if not candidates:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("没有满足 plausibility 的候选")  # 执行本行的状态、计算或校验逻辑。
    return max(candidates, key=lambda token: contrastive_score(token, expert, amateur, beta))  # 执行本行的状态、计算或校验逻辑。
chosen = choose_token(expert, amateur, 1.0, 0.4)  # 执行本行的状态、计算或校验逻辑。
assert chosen == "river"  # 执行本行的状态、计算或校验逻辑。
assert choose_token(expert, amateur, 0.0, 0.4) == "the"  # 执行本行的状态、计算或校验逻辑。
assert chosen in plausible  # 执行本行的状态、计算或校验逻辑。


## 5. 对齐：词表、tokenizer 与 chat template 任一不同时不可比较

概率字典的 key 等价于 token id。两个模型若采用不同 tokenizer、special token 或模板，`id=123` 并非同一个字符串；必须拒绝而不是表面上完成减法。


In [ ]:
def compatible_models(expert_meta, amateur_meta):  # 执行本行的状态、计算或校验逻辑。
    return expert_meta["tokenizer"] == amateur_meta["tokenizer"] and expert_meta["template"] == amateur_meta["template"]  # 执行本行的状态、计算或校验逻辑。
meta = {"tokenizer": "tok-v1", "template": "chat-v1"}  # 执行本行的状态、计算或校验逻辑。
assert compatible_models(meta, dict(meta))  # 执行本行的状态、计算或校验逻辑。
assert not compatible_models(meta, {**meta, "tokenizer": "tok-v2"})  # 执行本行的状态、计算或校验逻辑。
assert not compatible_models(meta, {**meta, "template": "chat-v2"})  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：零概率、空候选与重复 token 都要显式处理

真实 softmax 概率极少精确为零，但 mask、截断和非法 token 可能制造等价情况。教学实现拒绝非正概率；生产实现也要在 EOS、stop sequence、grammar mask 和安全过滤后重新验证候选集合。


In [ ]:
def validate_distribution(distribution):  # 执行本行的状态、计算或校验逻辑。
    return all(probability > 0 for probability in distribution.values()) and math.isclose(sum(distribution.values()), 1.0)  # 执行本行的状态、计算或校验逻辑。
assert validate_distribution(expert)  # 执行本行的状态、计算或校验逻辑。
assert not validate_distribution({"x": 1.0, "y": 0.0})  # 执行本行的状态、计算或校验逻辑。
assert not validate_distribution({"x": 0.8, "y": 0.1})  # 执行本行的状态、计算或校验逻辑。


## 7. 评测：流畅、多样、事实性与成本不能只看一种自动指标

CD 的目标是缓解开放生成退化，但它可能改变事实性、长度和延迟。报告应分开比较 expert baseline、CD、top-p 等策略，并使用固定 prompt、随机种子、人评或任务 verifier。


In [ ]:
def compare_methods(rows):  # 执行本行的状态、计算或校验逻辑。
    return {name: sum(item["quality"] for item in group) / len(group) for name, group in rows.items()}  # 执行本行的状态、计算或校验逻辑。
quality = compare_methods({"expert": [{"quality": 0.6}, {"quality": 0.7}], "cd": [{"quality": 0.8}, {"quality": 0.7}]})  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(quality["expert"], 0.65)  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(quality["cd"], 0.75)  # 执行本行的状态、计算或校验逻辑。
assert quality["cd"] > quality["expert"]  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：两个模型、beta、阈值与 prompt 集必须一起版本化

一条 CD 结果取决于两份权重、两个 tokenizer/template、plausibility alpha、beta 和 prompt/eval 集。只保存 expert checkpoint 无法复放，也无法解释结果是否来自 amateur 更新。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"expert": "large-v1", "amateur": "small-v1", "beta": 1.0, "plausibility": 0.4, "template": "chat-v1"}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["expert"] != artifact["amateur"]  # 执行本行的状态、计算或校验逻辑。
assert artifact["beta"] == 1.0  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明目标与状态，再给出核心公式、失败反例、独立评测和版本化制品。不要把对一个合成向量/几个候选的断言通过，误说成真实大模型上已经可靠、无偏或安全。
